# 🫀 퀘스트 46 · Q7-X — **진단 다발**: λ 가 어디서 오는지, null 이 왜 0.5 가 아닌지

| | **MedKOS / `notebooks/quest46_q7x_diagnostics.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-V**(λ 0.2586 · 자가 병목?) · **Q7-S″**(W5 ✅ · null 0.5090) |
| 규약 | **R11 · R16 · R22 · R24 · R29 ② · R30 ① · R33 ① · R34 ②③ · R35 ①④⑦ · R36 ①④⑤ · R37** |
| 학습 | **0회** · GPU 불필요 · **새 데이터 0** · 예상 20~35분 |

## 왜 이 런인가 — Q7-V 의 진단이 산수와 안 맞는다

Q7-V 요약은 「무딘 자는 분절기가 아니라 **±11ms 지터**」라고 썼다. 외부 검토가
산수로 반박했다 — P 파 반폭 40~50ms 에서 ±11ms 이동은 진폭을 **6% 남짓** 바꾼다.
그걸로 상관이 0.26(정보 74% 소실)까지 떨어질 수 없다.

**그런데 검토의 결론(「`p_score` 는 이미 창 통계량이니 창으로 바꿔도 안 오른다」)도
정의를 보면 성립하지 않는다:**

```python
p_score = |x_detrended(q)| / MAD(x[q-100ms : q+100ms])
          ↑ **단일 표본**       ↑ 창은 여기 — **스케일 정규화용**
```

**창은 분모에만 있다. 신호 측정은 한 점이다.** 그래서 「창인데도 0.26」이 아니라
**「점이라서 0.26」**일 수 있다.

그리고 그게 λ 0.26 을 정량적으로 설명한다. `x(q) = P(q) + n(q)` 에서 잡음이 P 대역
위에 있으면 `n(q)` 와 `n(q+Δ)` 는 금방 무상관이 되므로

```
corr(x(q), x(q+Δ)) ≈ ρ_P(Δ) · SNR/(1+SNR)      SNR ≈ 0.3 → ≈ 0.23
```

**세 가설이 λ(Δ) 곡선의 「모양」으로 깔끔히 갈린다:**

| λ(Δ) 모양 | 뜻 | Q7-Y 처방 |
|---|---|---|
| Δ=3ms 에서 이미 급락 후 **평탄** | **SNR 한계** — 점 통계량 자체 | 적분·매치드필터(잡음 평균) |
| Δ 따라 **완만히** 감소 | **지터 한계** | 이동 불변 표현 |
| Δ=11ms 에서도 **≥0.85** | 위치가 원인 **아님** | λ 0.26 의 출처를 다시 찾는다 |

그래서 λ(Δ) 를 **`p_score`(점 분자)와 `p_energy`(±40ms 적분 분자)** 두 통계량으로
나란히 그린다. 분모·창은 **동일**하게 두어 「점 vs 창」 축 하나만 바꾼다.

## 그 밖에 닫는 것 다섯

- **X0** λ 정의 감사 — `corr(true, obs)` 면 **나눗셈**이 맞고, test-retest 신뢰도면
  `√λ` 로 나눠야 한다(그러면 상한이 반토막). 코드 정의를 확인하고 **둘 다 찍는다**.
  덤으로 **고전오차 가정**(σ_obs > σ_true)을 검정한다 — 아니면 보정 방향이 다르다
- **X2** 레코드별 λ 히스토그램 — Se 평균 0.9109 vs 적중률 중앙 0.9914 의 **강한 좌편포**.
  소수 붕괴 레코드가 λ 를 끌어내리는가
- **X3 ★★** **쌍 내 치환 null** — S3 의 null 이 0.5090 인데 관문 문턱은 0.5 다.
  초과 +0.0272 의 **25%가 이 오프셋**이고, 다음 런에서 CI 하한 0.505 면 null 아래인데
  ✅ 가 찍힌다. 매칭 층 **안에서** 라벨을 교환하면 **구성상 0.5** 여야 한다
- **X4 ★★** 레코드 AUROC ~ **SVEB 부담** 회귀 — W3 에서 쌍 가중 추정치가 더 높았다
  (0.5646 vs 0.5362). 고부담 = 단일 우세 초점 = P 형태 일관 가설. 유의하면 다음 런의
  표적 모집단을 **여기서 사전등록**한다
- **X5 ★★** **동등분산 무작위 잡음 대조** — V2 는 「검출기 오차를 얹으면 죽는다」만
  봤다. **같은 크기 아무 잡음이나 죽인다면** V2 는 「검출기가 병목」의 증거가 아니라
  **「효과가 그만큼 약하다」**는 취약성 측정이다
- **X6** 의사결정 숫자 — 딥러닝 판단은 S3(기전)가 아니라 **W1(효용)** 로 한다

## 사전등록 — 관문

| 관문 | 판정 기준 |
|---|---|
| **X0** | λ 정의를 로그에 명시 · 나눗셈/√나눗셈 **둘 다** 병기 · 고전오차 가정 검정 |
| **X1 ★★★(주)** | λ(Δ=0) **정확히 1.0**(항등) · 모양으로 세 가설 판정 · `p_energy` 이득 |
| **X2** | 하위 10% 레코드를 빼면 평균 λ 가 **+0.15 이상** 오르나 |
| **X3 ★★** | 쌍 내 치환 null 이 **0.5±0.005** 인가 |
| **X4 ★★** | 부담 vs 레코드 AUROC 의 Spearman CI 가 0 을 배제하나 |
| **X5 ★★** | 동등분산 잡음이 검출기 오차와 **같은 정도로** 죽이나 |
| **X6** | W1 탈감쇠 **상한** (점추정 인용 금지 · R36 ①) |

### 판정표 (R29 ②)

- **X1 항등(Δ=0 ≠ 1.0)** → 점수 계산이 위치에 대해 결정적이지 않다. **중단**
- **X1 = 「위치 무관」** → Q7-V 의 「자가 병목」 해석을 **재검토**한다. λ 0.26 의 출처를
  X2(꼬리)·X0(고전오차 위배) 순으로 찾는다
- **X1 = 「SNR 한계」 + `p_energy` 이득 큼** → **Q7-Y 기본형 = 적분/매치드필터**
- **X1 = 「지터 한계」** → **Q7-Y 기본형 = 이동 불변 표현**(구간 최대 정규화 상호상관)
- **X3 이 0.5 아님** → S3 추정량에 버그다. **S3 전체를 다시 본다**
- **X5 에서 동등분산 잡음도 같이 죽인다** → V2 를 「자가 병목」의 증거로 **인용하지 않는다**.
  그 근거는 **V1(λ)+X1(λ 를 올릴 수 있나)** 만 남는다

⚠️ **이 런은 새 데이터를 쓰지 않고, SVEB 질문에도 답하지 않는다.** 도구를 잰다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    """**레코드 단위** 부트스트랩(R11)."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_stat(v, w, fn, seed, nb=2000, q=2.5):
    """두 배열에 대한 임의 통계량의 **레코드 부트스트랩**(짝을 유지한다)."""
    v = np.asarray(v, float); w = np.asarray(w, float)
    ok = np.isfinite(v) & np.isfinite(w)
    v, w = v[ok], w[ok]
    if len(v) < 5:
        return float("nan"), float("nan"), float("nan"), len(v)
    rng = np.random.RandomState(seed)
    b = []
    for _ in range(nb):
        j = rng.randint(0, len(v), len(v))
        try:
            b.append(float(fn(v[j], w[j])))
        except Exception:
            pass
    if len(b) < 50:
        return float(fn(v, w)), float("nan"), float("nan"), len(v)
    return (float(fn(v, w)), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(v))

def spearman(a, b):
    ra = np.asarray(a, float).argsort().argsort().astype(float)
    rb = np.asarray(b, float).argsort().argsort().astype(float)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300

# ── ★ Q7-V 와 **같은** 자 (감사 대상이 바뀌면 안 된다)
BUT_DIR = "but-pdb/1.0.0"
DELIN, INPUT = "dwt", "raw"
TOL_MS = 50.0
P_LO_MS, P_HI_MS = -278.0, -42.0
SCORE_HALF_MS = 100.0            # 분모(MAD) 창 반폭 — **두 통계량 공통**
SIG_HALF_MS   = 40.0             # ★ `p_energy` 의 **분자** 적분 반폭(P 폭 80~100ms)
MIN_P = 5

# ── ★★ X1 — λ(Δ) 격자. **Δ=0 은 항등**(정확히 1.0)
DELTA_MS = (0.0, 3.0, 5.0, 11.0, 20.0, 50.0, 100.0)
D_KEY    = 11.0                  # 판정 기준 Δ — Q7-V 의 적중 지터 MAD 11.1ms
D_SMALL  = 3.0                   # 「작은 Δ 에서 이미 낮은가」
LAM_HI, LAM_GAIN = 0.85, 0.20    # 위치 무관 문턱 · 적분 이득 문턱

# ── SVDB (Q7-S′·Q7-S″ 와 동일 상수 — 안 맞으면 재현이 깨진다)
QUANT_MS = 1000.0 / 128.0
FULL_K = tuple(range(4, 33))
MIN_S, MIN_N, MIN_PAIR = 25, 25, 200
TAIL_FRAC = 0.10                 # X2 — 하위 몇 %를 뺄까
TAIL_GAIN = 0.15                 # X2 판정 문턱
NULL_TOL  = 0.005                # X3 판정 문턱
N_PERM    = 20                   # 치환 반복

SV5  = os.path.join(MITBIH, "svdb_data5.npz")
PDEL = os.path.join(MITBIH, "svdb_pdelin.npz")

# ── ★ 앞선 공식 실행값 — 정합/재현 기준
REF = dict(
    u1=dict(se=0.9109, pp=0.7145, f1=0.7705),          # Q7-U `20260804T0458`
    s3=dict(auc=0.5362, null=0.5090, mde=0.0531, n=34),  # Q7-S′ `20260804T0658`
    w1=dict(mean=0.0048, lo=-0.0030, hi=0.0141, n=56),   # Q7-S″ `20260804T0809`
    lam_v=0.2586, a_z=0.2419, e_sd=0.9322)              # Q7-V `20260804T0834`
TOL_REPRO = 0.02

RULE_CHECK = {
    "R11 환자 단위":      "부트스트랩·판정 전부 **레코드 단위**",
    "R16 fallback 없음":  "BUT PDB·SVDB 자산이 없으면 **중단**",
    "R22 누수 없음":      "λ·잡음 분포는 BUT PDB 에서, SVDB 라벨은 안 본다",
    "R29 ② 분기 금지":    "항등(Δ=0)이 깨지면 어떤 결론도 읽지 않는다",
    "R30 ① 필요표본":     "X4 가 유의하면 부분군 필요표본을 같이 낸다",
    "R33 ① MDE":          "관문마다 MDE 를 내고 점추정과 비교",
    "R34 ② 선택 편의":    "★ X4 부분군은 **사전등록만** 하고 이 런에서 판정하지 않는다",
    "R34 ③ 대조 보장":    "★★ X5 — 동등분산 잡음은 **분산을 맞춰 구성**한다",
    "R35 ① 자 먼저":      "★ 이 런 전체가 **자를 재는** 일이다",
    "R35 ④ 항등 대조":    "★★ λ(Δ=0)=1.0 · 쌍 내 치환 null=0.5 — 둘 다 **구성으로** 보장",
    "R36 ① 상한":         "X6 탈감쇠는 **상한**으로만",
    "R36 ⑤ 성분 병기":    "λ 이득은 두 통계량 값과 **함께만** 인용",
    "R37 ① 프레임":       "필요표본은 **우월 프레임 · 검정력 명시**",
}

CONFIG = dict(
    exp="quest46_q7x_diagnostics", quest="ailab-2026-0046", step="lambda-diagnostics",
    parent_exp=["quest46_q7v_ruler_audit", "quest46_q7s3_recompute"],
    purpose=("Q7-V 의 진단(「무딘 자 = ±11ms 지터」)이 산수와 안 맞는다는 외부 지적을 "
             "**실측으로 가른다**. ★ 핵심 관찰: `p_score` 의 창은 **분모(MAD)에만** 있고 "
             "분자는 **단일 표본**이다 — 그래서 「창인데도 0.26」이 아니라 「점이라서 0.26」"
             "일 수 있다. λ(Δ) 곡선을 **점 분자 vs ±40ms 적분 분자** 두 통계량으로 그려 "
             "SNR 한계/지터 한계/위치 무관 셋을 가르고, 그 결과가 Q7-Y 설계를 정한다. "
             "덤으로 ① λ 정의 감사 ② 레코드별 λ 꼬리 ③ 쌍 내 치환 null(관문 문턱 교정) "
             "④ SVEB 부담 회귀(표적 모집단 사전등록) ⑤ 동등분산 잡음 대조 ⑥ 의사결정 숫자"),
    dataset="BUT PDB 50×2분(전문가 P 주석) + SVDB 78레코드 184,499비트",
    delineator=DELIN, input=INPUT, tol_ms=TOL_MS, p_win_ms=[P_LO_MS, P_HI_MS],
    score_half_ms=SCORE_HALF_MS, sig_half_ms=SIG_HALF_MS,
    delta_ms=list(DELTA_MS), d_key=D_KEY, d_small=D_SMALL,
    lam_hi=LAM_HI, lam_gain=LAM_GAIN, tail_frac=TAIL_FRAC, tail_gain=TAIL_GAIN,
    null_tol=NULL_TOL, n_perm=N_PERM, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "X0": "**λ 정의 감사.** 코드가 `corr(정답위치값, 검출위치값)` 이면 고전 측정오차에서 "
              "`corr = σ_t/σ_obs` 이므로 **λ 로 나누는 게 맞다**. `corr(측정1, 측정2)`(test-"
              "retest 신뢰도)라면 `√λ` 로 나눠야 하고 상한이 반토막 난다. 정의를 로그에 "
              "명시하고 **둘 다 찍는다**. ★ 덤: **고전오차 가정 검정**(σ_obs > σ_true 인가) — "
              "아니면 오차가 가법이 아니라 **수축**이고 보정 방향이 다르다",
        "X1": "★★★ **(주) λ(Δ) 곡선.** 정답 위치에서 **강제로 Δ 이동**시켜 다시 재고 "
              f"`λ(Δ) = corr(f@true, f@true+Δ)` 를 낸다. Δ={DELTA_MS} ms. "
              "**Δ=0 은 정확히 1.0**(항등 · 아니면 중단). 두 통계량으로 그린다 — "
              "`p_score`(분자가 **단일 표본**) vs `p_energy`(분자만 **±40ms RMS**, 분모·창 "
              "동일). 모양으로 판정: Δ=3ms 에서 이미 낮고 평탄 → **SNR 한계** · Δ 따라 완만 "
              f"감소 → **지터 한계** · Δ={D_KEY}ms 에서도 ≥{LAM_HI} → **위치 무관**",
        "X2": "레코드별 λ 히스토그램. Se 평균 0.9109 vs 적중률 중앙 0.9914 의 좌편포가 λ 를 "
              f"끌어내리는가 — 하위 {TAIL_FRAC:.0%} 레코드를 빼면 평균 λ 가 "
              f"**+{TAIL_GAIN}** 이상 오르나",
        "X3": "★★ **쌍 내 치환 null.** S3 의 null 이 0.5090 인데 관문 문턱은 0.5 다 — 초과 "
              "+0.0272 의 **25%가 이 오프셋**이고, 다음 런에서 CI 하한 0.505 면 null 아래인데 "
              "✅ 가 찍힌다. 매칭 **층 안에서** 라벨을 교환하면 **구성상 0.5** 여야 한다. "
              f"0.5±{NULL_TOL} 밖이면 **추정량 버그**이고 S3 전체를 다시 봐야 한다",
        "X4": "★★ SVEB **부담** vs 레코드 AUROC. W3 에서 쌍 가중 추정치가 더 높았다"
              "(0.5646 vs 0.5362) — 고부담 = 단일 우세 초점 = P 형태 일관 가설. Spearman CI 가 "
              "0 을 배제하면 다음 런의 표적 모집단을 **여기서 사전등록**한다. "
              "★ **이 런에서는 부분군을 판정하지 않는다**(사후 체리피킹 방지 · R34 ②)",
        "X5": "★★ **동등분산 무작위 잡음 대조.** V2 는 「검출기 오차를 얹으면 죽는다」만 "
              "봤다. 같은 순위상관까지 떨어뜨리는 **무작위** 잡음도 똑같이 죽인다면, V2 는 "
              "「검출기가 병목」의 증거가 아니라 **효과의 취약성 측정**이다. 그러면 "
              "「자가 병목」의 근거는 **V1(λ)+X1(λ 를 올릴 수 있나)** 만 남는다",
        "X6": "의사결정 숫자. 딥러닝 여부는 S3(기전)가 아니라 **W1(교차환자 증분 효용)** 로 "
              "판단해야 한다. W1 을 λ 로 탈감쇠한 **상한**을 낸다. ⚠️ 다변량 ΔAUPRC 에 "
              "단변량 λ 를 쓰는 건 조잡하다 — **상한으로만** 읽는다(R36 ①)"},
    caveat=("★ **새 데이터 0** · 학습 0회 · SVEB 질문에 답하지 않는다 — **도구를 잰다**. "
            "★ X1 은 **정답 위치를 강제 이동**시키므로 검출기와 무관하다 — 「점수 통계량이 "
            "위치에 얼마나 민감한가」만 잰다. 검출기 오차와 섞이지 않는 게 요점이다. "
            "★ X4 는 **가설 생성**이다 — 부분군 값을 이 런의 결론으로 쓰지 않는다. "
            "★ 표본율 전이: λ 는 BUT PDB 해상도에서 잰 값이고 SVDB 원본은 **128Hz(7.8ms 격자)**"
            "다. 지터 MAD 11.1ms 와 같은 자릿수라 SVDB 쪽엔 **격자가 정한 λ 하한**이 있다"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7x_diagnostics", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q7-X 진단 다발** — λ 가 어디서 오는지, null 이 왜 0.5 가 아닌지")
run.log(f"  ★★★ 주 관문 X1 = **λ(Δ) 곡선** · Δ={DELTA_MS} ms · 두 통계량")
run.log("       `p_score`(분자 **단일 표본**) vs `p_energy`(분자만 **±40ms RMS**)")
run.log("       → 분모·창은 **동일** — 「점 vs 창」 축 하나만 바꾼다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 【X-0】 BUT PDB 적재 (Q7-V 와 동일 경로)
try:
    import wfdb, neurokit2 as nk
except ImportError:
    !pip -q install wfdb neurokit2
    importlib.invalidate_caches(); import wfdb, neurokit2 as nk
import re, urllib.request
run.log("\n" + "=" * 100)
run.log("【X-0】 BUT PDB — 전문가 P 주석")
run.log("=" * 100)
BASE = f"https://physionet.org/files/{BUT_DIR}"
BEAT_SYM = set("NLRAaJSVFejE/fQ")

def _get(url, timeout=60):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace")

BUT_RECS = [str(r) for r in wfdb.get_record_list("but-pdb")]
if len(BUT_RECS) < 10:
    raise AssetError(f"BUT PDB 목록이 {len(BUT_RECS)}개 — 다운로드 실패(R16)")

def resolve_rid(r):
    for c in dict.fromkeys([r] + ([f"{int(r):0{w}d}" for w in (2, 3)] if r.isdigit() else [])):
        try:
            wfdb.rdheader(c, pn_dir=BUT_DIR); return c
        except Exception:
            continue
    return None

BUT_RECS = [c for c in (resolve_rid(r) for r in BUT_RECS) if c is not None]

def list_exts(rid):
    exts = []
    try:
        for ln in _get(f"{BASE}/ANNOTATORS").splitlines():
            tok = ln.split("\t")[0].strip() if ln.strip() else ""
            if tok and not tok.startswith("#"):
                exts.append(tok.split()[0])
    except Exception as e:
        run.log(f"  ⚠️ ANNOTATORS 실패: {type(e).__name__}")
    if not exts:
        try:
            html = _get(BASE + "/")
            exts = sorted({x for x in re.findall(rf"{re.escape(rid)}\.([A-Za-z0-9_]+)", html)
                           if x not in ("dat", "hea", "xws", "png", "txt")})
        except Exception as e:
            run.log(f"  ⚠️ 디렉터리 목록 실패: {type(e).__name__}")
    return exts

PROBE = []
for e in list_exts(BUT_RECS[0]):
    try:
        a = wfdb.rdann(BUT_RECS[0], e, pn_dir=BUT_DIR)
        PROBE.append((e, len(a.sample), sorted(set(a.symbol))))
    except Exception:
        pass
if not PROBE:
    raise AssetError(f"{BUT_RECS[0]}: 읽히는 주석이 없다")
EXT_P = next((e for e, _, _ in PROBE if e.lower().startswith("p")), None)
_rest = [(e, n_, sy) for e, n_, sy in PROBE if e != EXT_P]
EXT_Q = max(_rest, key=lambda t: (len(set(t[2]) & BEAT_SYM), t[1]))[0] if _rest else None
if EXT_P is None or EXT_Q is None:
    raise AssetError(f"주석 역할을 못 가렸다 — {[(e, n) for e, n, _ in PROBE]}")

BUT, T0 = {}, time.time()
for rid in BUT_RECS:
    rec = wfdb.rdrecord(rid, pn_dir=BUT_DIR)
    sig = np.nan_to_num(np.asarray(rec.p_signal, float), nan=0.0, posinf=0.0, neginf=0.0)
    rp = np.asarray(wfdb.rdann(rid, EXT_Q, pn_dir=BUT_DIR).sample, int)
    pp = np.asarray(wfdb.rdann(rid, EXT_P, pn_dir=BUT_DIR).sample, int)
    if len(rp) < 5 or len(pp) < MIN_P:
        continue
    BUT[rid] = dict(sig=sig[:, :2], fs=int(rec.fs), r=rp, p=pp)
_fs = sorted({v["fs"] for v in BUT.values()})
run.log(f"  적재 {len(BUT)}개 · {time.time()-T0:.0f}초 · 주석 QRS `{EXT_Q}` · P `{EXT_P}`")
run.log(f"  표본율 {_fs}Hz — ⚠️ SVDB 원본은 **128Hz(7.8ms 격자)**. 지터 MAD 11.1ms 와 같은")
run.log("     자릿수라 SVDB 쪽에는 **격자가 정한 λ 하한**이 있다(전이 가정)")
CONFIG["but"] = dict(n_rec=len(BUT), fs=[int(f) for f in _fs])
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【X-A】 두 점수 통계량 + X0 λ 정의 감사
run.log("\n" + "=" * 100)
run.log("【X-A】 X0 — λ 정의 감사 · 두 점수 통계량 정의")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def ms2s(ms, fs):
    return int(round(ms * fs / 1000.0))

def _detrend(v):
    n = len(v)
    if n < 3:
        return np.asarray(v, float)
    t = np.arange(n, dtype=float)
    a, b = np.polyfit(t, np.asarray(v, float), 1)
    return np.asarray(v, float) - (a * t + b)

def _win(x, q, fs):
    """공통 창 — 위치 중심 ±SCORE_HALF_MS. 두 통계량이 **같은 창·같은 분모**를 쓴다."""
    w = ms2s(SCORE_HALF_MS, fs)
    a, b = max(int(q) - w, 0), min(int(q) + w + 1, len(x))
    if b - a < 5:
        return None, None, None
    seg = _detrend(x[a:b])
    den = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
    return seg, int(q) - a, den

def score_point(x, pos, fs):
    """Q7-P0·Q7-V 의 `p_score` — ★ **분자가 단일 표본**이다."""
    out = []
    for q in np.atleast_1d(pos):
        seg, c, den = _win(x, q, fs)
        out.append(0.0 if seg is None else float(abs(seg[c]) / den))
    return np.asarray(out, float)

def score_energy(x, pos, fs):
    """★ **분자만** ±SIG_HALF_MS 구간 RMS 로 바꾼 대안. 분모·창은 위와 **동일**.

    ★★ 이 한 축이 이 런의 요점이다 — `p_score` 의 창은 **분모에만** 있고 신호 측정은
      한 점이다. 적분하면 잡음이 평균되므로, λ 가 SNR 한계면 여기서 회복되고
      순수 지터 한계면 덜 회복된다."""
    h = ms2s(SIG_HALF_MS, fs)
    out = []
    for q in np.atleast_1d(pos):
        seg, c, den = _win(x, q, fs)
        if seg is None:
            out.append(0.0); continue
        a, b = max(c - h, 0), min(c + h + 1, len(seg))
        out.append(float(np.sqrt(np.mean(seg[a:b] ** 2)) / den))
    return np.asarray(out, float)

STATS = {"p_score": score_point, "p_energy": score_energy}
run.log(f"  `p_score`  = |x_detrend(q)| / MAD(±{SCORE_HALF_MS:.0f}ms)   ← 분자 **단일 표본**")
run.log(f"  `p_energy` = RMS(x_detrend[q±{SIG_HALF_MS:.0f}ms]) / MAD(±{SCORE_HALF_MS:.0f}ms)"
        "   ← 분자만 **적분**")
run.log("  ▸ 분모·창이 같으므로 두 값의 차이는 **분자가 점이냐 창이냐** 하나뿐이다")

# ── X0 — λ 정의 감사
run.log("\n  ★ X0 — λ 정의 감사")
run.log("    코드의 λ 는 `corr(f@정답위치, f@검출위치)` 다 — 한쪽이 **기준 표준**이다.")
run.log("    고전 측정오차 `X_obs = X_true + e` (e ⊥ X_true) 에서")
run.log("      corr(X_true, X_obs) = σ_t/σ_obs,   d_obs = Δμ/σ_obs,  d_true = Δμ/σ_t")
run.log("      → **d_true = d_obs / λ**  (나눗셈이 맞다)")
run.log("    `corr(측정1, 측정2)`(둘 다 불완전 = test-retest 신뢰도 ρ = σ²_t/σ²_obs)라면")
run.log("      σ_t/σ_obs = √ρ 이므로 **√λ 로 나눠야** 하고 상한이 반토막 난다.")
run.log("    → 우리 정의는 **전자**다. 그래도 아래에서 **둘 다** 찍는다")

In [ ]:
# CELL 4 — 【X-B】 ★★★ X1 λ(Δ) 곡선 + X0 고전오차 검정 + X2 레코드 꼬리
run.log("\n" + "=" * 100)
run.log("【X-B】 X1(주) — λ(Δ) 곡선 · 두 통계량 · **Δ=0 은 항등**")
run.log("=" * 100)
run.log("  ▸ **정답 위치를 강제로 Δ 이동**시켜 다시 잰다 — 검출기와 무관하다.")
run.log("    「점수 통계량이 위치에 얼마나 민감한가」만 재는 게 요점이다")

def true_p_positions(rid):
    """탐색창 안에 있는 **정답 P** 만 — 자가 볼 수 있는 모집단으로 한정."""
    d = BUT[rid]; fs = d["fs"]
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    out = []
    for pt in d["p"]:
        nxt = d["r"][d["r"] > pt]
        if not len(nxt):
            continue
        R = int(nxt[0])
        if R + lo <= pt < R + hi:
            out.append(int(pt))
    return np.asarray(out, int)

T1_ = time.time()
LAM = {s: {dm: [] for dm in DELTA_MS} for s in STATS}
SIGMA = {s: [] for s in STATS}                 # (σ_true, σ_obsΔ) — 고전오차 검정
RIDS = sorted(BUT)
for rid in RIDS:
    d = BUT[rid]; fs = d["fs"]; x = d["sig"][:, 0]
    pt = true_p_positions(rid)
    if len(pt) < 20:
        continue
    for sname, fn in STATS.items():
        v0 = fn(x, pt, fs)
        if np.std(v0) < 1e-12:
            continue
        for dm in DELTA_MS:
            sh = ms2s(dm, fs)
            if sh == 0:
                LAM[sname][dm].append(1.0)      # ★ 구성상 항등
                continue
            # ★ **양방향 평균** — 편향된 한쪽 이동만 보면 비대칭 파형에서 왜곡된다
            cs = []
            for sgn in (+1, -1):
                q = np.clip(pt + sgn * sh, 0, len(x) - 1)
                v1 = fn(x, q, fs)
                if np.std(v1) > 1e-12:
                    cs.append(float(np.corrcoef(v0, v1)[0, 1]))
            if cs:
                LAM[sname][dm].append(float(np.mean(cs)))
            if dm == D_KEY and sname == "p_score":
                q = np.clip(pt + sh, 0, len(x) - 1)
                SIGMA[sname].append((float(np.std(v0)), float(np.std(fn(x, q, fs)))))
run.log(f"  ({time.time()-T1_:.0f}초) λ 를 잰 레코드 "
        f"{len(LAM['p_score'][D_KEY])} · 정답 P 기준")

# ── ★ 항등 대조 — Δ=0 은 정확히 1.0
for s in STATS:
    z = np.asarray(LAM[s][0.0], float)
    if len(z) and float(np.max(np.abs(z - 1.0))) > 1e-12:
        raise AssetError(f"{s}: λ(Δ=0) 이 1.0 이 아니다 — 점수 계산이 위치에 대해 "
                         "결정적이지 않다. 이 상태로는 곡선 전체가 무의미하다")
run.log("  ★ 항등 대조 — λ(Δ=0) = 1.000000 (두 통계량 모두 · 구성으로 보장)")

run.log(f"\n  {'Δ(ms)':>7}{'p_score':>10}{'CI':>20}{'p_energy':>11}{'CI':>20}{'이득':>9}")
X1 = {}
for dm in DELTA_MS:
    row = {}
    for s in STATS:
        m_, lo_, hi_, n_ = boot_mean(LAM[s][dm], SEED0 + 11 + int(dm))
        row[s] = dict(lam=m_, lo=lo_, hi=hi_, n=n_)
    gain = row["p_energy"]["lam"] - row["p_score"]["lam"]
    row["gain"] = float(gain)
    X1[dm] = row
    run.log(f"  {dm:>7.0f}{row['p_score']['lam']:>10.4f}"
            f"  [{row['p_score']['lo']:>6.4f},{row['p_score']['hi']:>7.4f}]"
            f"{row['p_energy']['lam']:>11.4f}"
            f"  [{row['p_energy']['lo']:>6.4f},{row['p_energy']['hi']:>7.4f}]"
            f"{gain:>+9.4f}")

lk = X1[D_KEY]["p_score"]["lam"]; ls_ = X1[D_SMALL]["p_score"]["lam"]
gain_k = X1[D_KEY]["gain"]
if lk >= LAM_HI:
    shape = "위치 무관"
elif ls_ >= LAM_HI:
    shape = "지터 한계"
else:
    shape = "SNR 한계"
X1["shape"] = shape
g_("X1", "✅ 지지" if shape != "위치 무관" else "❌ 기각",
   f"★★★ λ(Δ={D_SMALL:.0f}ms) {ls_:.4f} · λ(Δ={D_KEY:.0f}ms) {lk:.4f} → **{shape}** · "
   f"적분 이득 @Δ={D_KEY:.0f}ms **{gain_k:+.4f}**")
run.log(f"    ▸ Q7-V 실측 λ(검출기 기반) = {REF['lam_v']:.4f} 와 비교하라 —")
run.log(f"      λ(Δ={D_KEY:.0f}ms)={lk:.4f} 가 그것보다 **훨씬 높으면** 위치 이동만으로는")
run.log("      Q7-V 의 λ 를 설명할 수 없다는 뜻이다(→ X2 꼬리 · X0 고전오차 위배를 본다)")
if shape == "SNR 한계":
    run.log("    ▸ **SNR 한계** — 작은 Δ 에서 이미 낮다. 점 통계량 자체가 잡음에 지배된다.")
    run.log(f"      Q7-Y 처방 = **적분·매치드필터**(이득 {gain_k:+.4f} 가 그 예고편)")
elif shape == "지터 한계":
    run.log("    ▸ **지터 한계** — 작은 Δ 에선 높고 Δ 따라 떨어진다.")
    run.log("      Q7-Y 처방 = **이동 불변 표현**(구간 최대 정규화 상호상관)")
else:
    run.log("    ▸ **위치 무관** — 위치를 흔들어도 λ 가 안 떨어진다. Q7-V 의 λ 0.2586 은")
    run.log("      위치에서 온 게 아니다. **Q7-Y 를 짓기 전에 출처를 먼저 찾아야 한다**")
g_("X1b", "✅ 지지" if gain_k >= LAM_GAIN else "⚠️ 미결",
   f"적분 분자가 λ 를 **{gain_k:+.4f}** 회복시킨다 (문턱 {LAM_GAIN}) — "
   f"성분 `p_score` {lk:.4f} · `p_energy` {X1[D_KEY]['p_energy']['lam']:.4f} (R36 ⑤)")

# ── X0 계속 — 고전오차 가정 검정
sg = np.asarray(SIGMA["p_score"], float)
if len(sg) >= 5:
    ratio = sg[:, 1] / (sg[:, 0] + 1e-12)
    rm, rlo, rhi, rn = boot_mean(ratio, SEED0 + 19)
    run.log(f"\n  ★ X0 고전오차 검정 — σ_obs/σ_true = **{rm:.4f}** [{rlo:.4f}, {rhi:.4f}] "
            f"(Δ={D_KEY:.0f}ms · n={rn})")
    cls_ok = rlo >= 0.98
    g_("X0", "✅ 지지" if cls_ok else "❌ 기각",
       ("가법(고전) 오차와 정합 — σ_obs ≥ σ_true 이므로 **λ 로 나누는** 보정이 맞다"
        if cls_ok else
        "★ **σ_obs < σ_true — 가법 오차가 아니라 수축이다.** 고전 보정을 그대로 쓰면 "
        "안 되고, 탈감쇠 상한을 재검토해야 한다"))
    CONFIG["X0"] = dict(sigma_ratio=rm, lo=rlo, hi=rhi, classical=bool(cls_ok))
else:
    g_("X0", "⛔ 측정 불가", "σ 비를 잴 레코드가 부족하다")

# ── X2 — 레코드별 λ 꼬리
run.log(f"\n  ★ X2 — 레코드별 λ 분포 (Δ={D_KEY:.0f}ms · `p_score`)")
lv = np.asarray(LAM["p_score"][D_KEY], float)
lv = lv[np.isfinite(lv)]
if len(lv) >= 10:
    cut = int(np.ceil(TAIL_FRAC * len(lv)))
    trimmed = np.sort(lv)[cut:]
    gain2 = float(trimmed.mean() - lv.mean())
    run.log(f"    전체 평균 {lv.mean():.4f} · 중앙 {np.median(lv):.4f} · "
            f"최소 {lv.min():.4f} · 하위 {TAIL_FRAC:.0%}({cut}개) 제거 후 {trimmed.mean():.4f}")
    g_("X2", "✅ 지지" if gain2 >= TAIL_GAIN else "❌ 기각",
       (f"★ **소수 붕괴 레코드가 λ 를 끌어내린다** (+{gain2:.4f} ≥ {TAIL_GAIN})"
        if gain2 >= TAIL_GAIN else
        f"꼬리를 잘라도 λ 가 +{gain2:.4f} 밖에 안 오른다 — **꼬리 탓이 아니다**"))
    CONFIG["X2"] = dict(mean=float(lv.mean()), med=float(np.median(lv)),
                        trimmed=float(trimmed.mean()), gain=gain2,
                        per_rec=[float(v) for v in lv])
CONFIG["X1"] = {str(k): v for k, v in X1.items()}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【X-C】 SVDB 적재 · 특징 · S3 재현 (Q7-S′ 와 동일)
import pandas as pd
from scipy.stats import norm
run.log("\n" + "=" * 100)
run.log("【X-C】 SVDB — Q7-S′ 특징 재구성 + S3 재현")
run.log("=" * 100)
for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "Q7-P0")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True); PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); SYM = np.asarray(D5["sym"]).astype(str)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
if int((np.asarray(PD["pid"]).astype(int) != PID).sum()) or \
   int((np.asarray(PD["sym"]).astype(str) != SYM).sum()):
    raise AssetError("정합 깨짐 — Q7-P0 를 다시 돌린다")
P_IDX = np.asarray(PD["p_idx"]).astype(int); P_SC = np.asarray(PD["p_score"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx0 = P_IDX[K].copy(); psc0 = P_SC[K].copy()
RS = np.array(sorted(set(RID.tolist())))
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    X = basis_ext(idx); y = v[idx]
    okm = np.isfinite(y)
    if okm.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[okm], y[okm], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx, perm=None, rng=None):
    """`perm` — None(관측) · 'record'(Q7-S′ 방식) · 'stratum'(★ 쌍 내 치환)."""
    tt = TT[idx]; key = np.round(f1[idx]).astype(int)
    if perm == "record":
        tt = tt.copy()
        for u in np.unique(RID[idx]):
            m = np.where(RID[idx] == u)[0]
            tt[m] = tt[m][rng.permutation(len(m))]
    elif perm == "stratum":
        tt = tt.copy()
        for kk in np.unique(key):                      # ★ **매칭 층 안에서** 교환
            m = np.where(key == kk)[0]
            tt[m] = tt[m][rng.permutation(len(m))]
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

REC_OK = [r for r in RS
          if (lambda i: TT[i].sum() >= MIN_S and (~TT[i]).sum() >= MIN_N)(np.where(RID == r)[0])]
PER, NPR, RID_OK = [], [], []
for r in REC_OK:
    idx = np.where(RID == r)[0]
    a, npr = matched_auc(resid(psc0, idx), idx)
    if np.isfinite(a):
        PER.append(a); NPR.append(npr); RID_OK.append(r)
m3, l3, h3, n3 = boot_mean(PER, SEED0 + 51)
d_ref = abs(m3 - REF["s3"]["auc"])
run.log(f"  비트 {len(K):,} · 레코드 {len(RS)} · 판정 후보 {len(REC_OK)} · 매칭 성립 {n3}")
run.log(f"  S3 재현 — {m3:.4f} [{l3:.4f}, {h3:.4f}] vs Q7-S′ {REF['s3']['auc']:.4f} · "
        f"|Δ| {d_ref:.6f}")
if d_ref > TOL_REPRO:
    raise AssetError(f"S3 재현 실패(|Δ| {d_ref:.4f} > {TOL_REPRO}) — 자산이나 환경이 "
                     "바뀌었고, 그러면 아래 진단을 Q7-S′ 와 나란히 놓을 수 없다")
run.log("  ✅ 재현 — 아래 진단을 Q7-S′·Q7-S″ 와 나란히 놓을 수 있다")
CONFIG["svdb"] = dict(n=int(len(K)), n_rec=int(len(RS)), n_ok=int(len(REC_OK)),
                      s3=float(m3), s3_lo=float(l3), s3_hi=float(h3), n_match=int(n3))
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【X-D】 ★★ X3 쌍 내 치환 null · X4 부담 회귀
run.log("\n" + "=" * 100)
run.log("【X-D】 X3 — 쌍 내 치환 null · X4 — SVEB 부담 회귀")
run.log("=" * 100)
run.log("  ▸ S3 의 null 이 0.5090 인데 관문 문턱은 **0.5** 다. 초과 +0.0272 의 25%가")
run.log("    이 오프셋이고, 다음 런에서 CI 하한 0.505 면 **null 아래인데 ✅** 가 찍힌다")
run.log("  ▸ 매칭 **층 안에서** 라벨을 교환하면 층 구조가 보존되므로 **구성상 0.5** 다.")
run.log("    Q7-S′ 의 셔플은 **레코드 안**에서 했다 — 층 크기·불균형이 안 보존된다")

T2_ = time.time()
NUL = {"record": [], "stratum": []}
for r in RID_OK:
    idx = np.where(RID == r)[0]
    vr = resid(psc0, idx)
    for mode in NUL:
        vals = []
        for s in range(N_PERM):
            rng = np.random.RandomState(SEED0 + 700 + 13 * s)
            a, _ = matched_auc(vr, idx, perm=mode, rng=rng)
            if np.isfinite(a):
                vals.append(a)
        if vals:
            NUL[mode].append(float(np.mean(vals)))
run.log(f"\n  ({time.time()-T2_:.0f}초) {'치환 방식':<16}{'null':>10}{'CI':>22}{'0.5 와의 차':>12}")
X3 = {}
for mode, lab in (("record", "레코드 안(옛)"), ("stratum", "★ 쌍 내(새)")):
    m_, lo_, hi_, n_ = boot_mean(NUL[mode], SEED0 + 61)
    X3[mode] = dict(null=m_, lo=lo_, hi=hi_, n=n_, dev=float(abs(m_ - 0.5)))
    run.log(f"  {lab:<16}{m_:>10.4f}  [{lo_:>7.4f}, {hi_:>7.4f}]{abs(m_-0.5):>12.4f}")
sn = X3["stratum"]
ok3 = sn["dev"] <= NULL_TOL
g_("X3", "✅ 지지" if ok3 else "❌ 기각",
   (f"★★ 쌍 내 치환 null = **{sn['null']:.4f}** (0.5±{NULL_TOL} 안) — 옛 오프셋 "
    f"{X3['record']['null']:.4f} 는 **셔플 방식 탓**이었다. 이걸 새 null 로 채택한다"
    if ok3 else
    f"★★ 쌍 내 치환인데도 **{sn['null']:.4f}** 다 — 층 구조를 보존해도 0.5 가 안 나온다. "
    "**추정량에 버그가 있다**(동점 처리·레코드 가중·층 불균형). S3 전체를 다시 봐야 한다"))
NEW_NULL = sn["null"] if np.isfinite(sn["null"]) else 0.5
run.log(f"    ▸ ★ **관문 문턱을 0.5 가 아니라 측정된 null({NEW_NULL:.4f})로** 놓는다.")
exc_old = m3 - REF["s3"]["null"]; exc_new = m3 - NEW_NULL
run.log(f"      초과 재계산 — 옛 null 기준 {exc_old:+.4f} · 새 null 기준 **{exc_new:+.4f}**")
half3 = mde(l3, h3)
for nm, ex in (("옛 null", exc_old), ("새 null", exc_new)):
    if ex > 0:
        s50 = n3 * (half3 / ex) ** 2
        run.log(f"      {nm} — 우월 필요 매칭레코드 50% {s50:.0f} · 80% {s50*2.04:.0f} "
                f"(총 환산 {s50/(n3/len(REC_OK)):.0f} · {s50*2.04/(n3/len(REC_OK)):.0f})")

# ── X4 — 부담 회귀 (가설 생성 · 이 런에서 판정하지 않는다)
run.log("\n  ★ X4 — 레코드 AUROC ~ SVEB 부담 · 쌍 수 · 신호품질  (**가설 생성** · R34 ②)")
BURD, QUAL = [], []
for r in RID_OK:
    idx = np.where(RID == r)[0]
    BURD.append(float(TT[idx].mean()))
    nm_ = idx[(~TT[idx]) & (pidx0[idx] >= 0)]
    QUAL.append(float(np.median(psc0[nm_])) if len(nm_) >= 10 else np.nan)
PERa = np.asarray(PER, float)
X4 = {}
run.log(f"    {'공변량':<12}{'Spearman':>10}{'CI':>22}{'n':>5}")
for nm, cv in (("SVEB 부담", np.asarray(BURD, float)),
               ("쌍 수(log)", np.log1p(np.asarray(NPR, float))),
               ("신호품질", np.asarray(QUAL, float))):
    rho, lo_, hi_, n_ = boot_stat(cv, PERa, lambda a, b: spearman(a, b), SEED0 + 71)
    X4[nm] = dict(rho=rho, lo=lo_, hi=hi_, n=n_)
    run.log(f"    {nm:<12}{rho:>10.4f}  [{lo_:>7.4f}, {hi_:>7.4f}]{n_:>5}")
bd = X4["SVEB 부담"]
sig4 = np.isfinite(bd["lo"]) and (bd["lo"] > 0 or bd["hi"] < 0)
g_("X4", "✅ 지지" if sig4 else "⚠️ 미결",
   (f"★★ 부담과 레코드 AUROC 가 연관된다(ρ {bd['rho']:+.4f}) — **다음 런의 표적 모집단을 "
    "여기서 사전등록한다**" if sig4 else
   f"부담 연관 미결(ρ {bd['rho']:+.4f} · CI 가 0 을 덮는다) — 모집단을 바꿀 근거가 없다"))
if sig4:
    hi_b = PERa[np.asarray(BURD) >= np.median(BURD)]
    lo_b = PERa[np.asarray(BURD) < np.median(BURD)]
    run.log(f"    (탐색적 · **판정 아님**) 부담 상위 절반 AUROC {np.mean(hi_b):.4f} "
            f"(n={len(hi_b)}) vs 하위 절반 {np.mean(lo_b):.4f} (n={len(lo_b)})")
    run.log("    ★★ **사전등록** — 다음 런의 표적 모집단은 「그 코호트의 SVEB 부담 중앙값")
    run.log("       이상」이다. 문턱은 **라벨을 보기 전에** 코호트 부담 분포로만 정한다.")
    run.log("       근거: 고부담 = 단일 우세 초점 = P 형태 일관 · 임상 표적(PAC 부담→AF)과 일치")
else:
    run.log("    ▸ 유의하지 않으므로 **표적 모집단을 바꾸지 않는다**(R34 ②)")
CONFIG["X3"] = X3; CONFIG["X4"] = X4; CONFIG["new_null"] = float(NEW_NULL)
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【X-E】 ★★ X5 동등분산 잡음 대조 · X6 의사결정 숫자
run.log("\n" + "=" * 100)
run.log("【X-E】 X5 — 동등분산 무작위 잡음 대조 · X6 — 의사결정 숫자")
run.log("=" * 100)
run.log("  ▸ V2 는 「검출기 오차를 얹으면 죽는다」만 봤다. 물어야 할 건")
run.log("    **「같은 크기 아무 잡음이나 죽이는가」**다. 죽인다면 V2 는 「검출기가 병목」의")
run.log("    증거가 아니라 **효과의 취약성 측정**이고, 「자가 병목」의 근거는")
run.log("    **V1(λ)+X1(λ 를 올릴 수 있나)** 만 남는다")

def transport(sc, rid, rng, a, e):
    """Q7-V 와 **같은** 순위공간 수송. a=1·e=0 이면 항등(구성 보장)."""
    out = np.array(sc, float).copy()
    for u in np.unique(rid):
        m = np.where(rid == u)[0]
        x = out[m]; mm = len(x)
        if mm < 5:
            continue
        r = x.argsort().argsort().astype(float) + 1.0
        z = norm.ppf(r / (mm + 1.0))
        zp = a * z + rng.choice(e, mm)
        rp = np.clip(np.round(norm.cdf(zp) * (mm + 1.0)).astype(int), 1, mm)
        out[m] = np.sort(x)[rp - 1]
    return out

def s3_of(vec):
    per = []
    for r in RID_OK:
        idx = np.where(RID == r)[0]
        a, _ = matched_auc(resid(vec, idx), idx)
        if np.isfinite(a):
            per.append(a)
    return boot_mean(per, SEED0 + 81)

# ── 두 팔의 **순위상관 감쇠를 같게** 맞춘다 (R34 ③ — 구성으로 보장)
A_DET, E_SD = REF["a_z"], REF["e_sd"]                 # Q7-V 실측 검출기 수송
rho_det = A_DET / np.sqrt(A_DET ** 2 + E_SD ** 2)     # 이 수송이 남기는 순위상관
sd_gen = float(np.sqrt(max(1.0 / rho_det ** 2 - 1.0, 1e-9)))   # z'=z+η 로 같은 ρ 를 내는 η
run.log(f"\n  검출기 수송 — a {A_DET:.4f} · e_sd {E_SD:.4f} → 남는 순위상관 ρ **{rho_det:.4f}**")
run.log(f"  동등분산 잡음 — z' = z + η · sd(η) **{sd_gen:.4f}** → 같은 ρ {1/np.sqrt(1+sd_gen**2):.4f}")
run.log("  ▸ **두 팔이 같은 ρ 를 내도록 구성했다** — 차이가 있다면 그건 크기가 아니라 **구조**다")

_rng = np.random.RandomState(SEED0 + 909)
E_DET = _rng.normal(0.0, E_SD, 20000)
E_GEN = _rng.normal(0.0, sd_gen, 20000)
ARMS5 = (("검출기 오차(V2)", A_DET, E_DET), ("동등분산 무작위", 1.0, E_GEN))
X5 = {"k0": float(m3)}
run.log(f"\n  {'팔':<18}{'k=0':>10}{'k=1':>10}{'초과 변화':>12}")
for nm, a_, e_ in ARMS5:
    reps = []
    for rep in range(6):
        rng = np.random.RandomState(SEED0 + 950 + rep)
        reps.append(s3_of(transport(psc0, RID, rng, a_, e_))[0])
    v1 = float(np.nanmean(reps))
    X5[nm] = dict(k1=v1, drop=float(m3 - v1), sd=float(np.nanstd(reps)))
    run.log(f"  {nm:<18}{m3:>10.4f}{v1:>10.4f}{v1-m3:>+12.4f}")
d_det = X5["검출기 오차(V2)"]["drop"]; d_gen = X5["동등분산 무작위"]["drop"]
same5 = abs(d_det - d_gen) <= 0.3 * max(abs(d_det), 1e-9)
g_("X5", "❌ 기각" if same5 else "✅ 지지",
   (f"★★ **동등분산 잡음도 똑같이 죽인다**(검출기 {d_det:+.4f} vs 무작위 {d_gen:+.4f}) — "
    "V2 는 「검출기가 병목」의 증거가 **아니라 효과의 취약성 측정**이다"
    if same5 else
    f"검출기 오차가 **구조적으로 더/덜** 해롭다(검출기 {d_det:+.4f} vs 무작위 {d_gen:+.4f}) — "
    "V2 를 검출기 특이적 증거로 읽을 수 있다"))
if same5:
    run.log("    ▸ 그러면 「자가 병목」의 근거는 **V1(λ 가 낮다) + X1(λ 를 올릴 수 있다)**")
    run.log("      뿐이다. V2 의 기울기는 **인용하지 않는다**")

# ── X6 — 의사결정 숫자
run.log("\n  ★ X6 — 의사결정 숫자 (딥러닝 판단은 S3 가 아니라 **W1** 로 · R37)")
w1 = REF["w1"]; lam = X1[D_KEY]["p_score"]["lam"]
lam_v = REF["lam_v"]
def deatt_lin(v, l):
    return float(v / max(min(l, 1.0), 1e-6))
X6 = {}
run.log(f"    {'λ 출처':<22}{'λ':>8}{'W1 상한(÷λ)':>14}{'√λ 대안':>11}")
for nm, l_ in (("Q7-V 검출기 기반", lam_v), (f"X1 위치 Δ={D_KEY:.0f}ms", lam)):
    up = deatt_lin(w1["hi"], l_)
    up_s = deatt_lin(w1["hi"], np.sqrt(max(l_, 1e-9)))
    X6[nm] = dict(lam=float(l_), upper=up, upper_sqrt=up_s)
    run.log(f"    {nm:<22}{l_:>8.4f}{up:>14.4f}{up_s:>11.4f}")
run.log(f"    ▸ W1 관측 {w1['mean']:+.4f} [{w1['lo']:+.4f}, {w1['hi']:+.4f}] · "
        "리듬 기저 AUPRC 0.416")
run.log("    ⚠️ **상한으로만 읽는다** — 다변량 ΔAUPRC 에 단변량 λ 를 쓰는 건 조잡하다(R36 ①)")
run.log("    ⚠️ **딥러닝 판단은 이 숫자로 한다** — S3 의 탈감쇠(+0.1038)는 **기전**의 크기이지")
run.log("       **효용**의 크기가 아니다")
CONFIG["X5"] = X5; CONFIG["X6"] = X6
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【X-F】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① ★★★ λ(Δ) 곡선 — 이 런의 심장
dd = list(DELTA_MS)
for s, c, mk in (("p_score", "tab:red", "o"), ("p_energy", "tab:blue", "s")):
    y = [X1[d][s]["lam"] for d in dd]
    lo = [X1[d][s]["lo"] for d in dd]; hi = [X1[d][s]["hi"] for d in dd]
    ax[0].plot(dd, y, mk + "-", color=c, label=s)
    ax[0].fill_between(dd, lo, hi, color=c, alpha=.15)
ax[0].axhline(REF["lam_v"], ls="--", color="k", lw=1.0)
ax[0].annotate(f"Q7-V lambda={REF['lam_v']:.3f}", (max(dd), REF["lam_v"]),
               fontsize=7, ha="right", va="bottom")
ax[0].axvline(D_KEY, ls=":", color="tab:gray", lw=1.0)
ax[0].set_ylim(0, 1.02)
ax[0].set_xlabel("forced shift from TRUE P position  |delta|  (ms)")
ax[0].set_ylabel("lambda = corr(f@true, f@true+delta)")
ax[0].set_title(f"X1 : {X1['shape']}", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② null 비교 + 레코드별 λ 히스토그램(쌍축)
nb_ = ["record-shuffle\n(old)", "within-stratum\n(new)"]
nv = [X3["record"]["null"], X3["stratum"]["null"]]
ne = [[nv[i] - X3[k]["lo"] for i, k in enumerate(("record", "stratum"))],
      [X3[k]["hi"] - nv[i] for i, k in enumerate(("record", "stratum"))]]
ax[1].bar(range(2), [v - 0.5 for v in nv], 0.5,
          color=["tab:gray", "tab:green"], bottom=0.5)
ax[1].errorbar(range(2), nv, yerr=ne, fmt="none", ecolor="k", capsize=5)
ax[1].axhline(0.5, color="k", lw=1.0)
ax[1].axhspan(0.5 - NULL_TOL, 0.5 + NULL_TOL, color="tab:green", alpha=.15)
ax[1].set_xticks(range(2)); ax[1].set_xticklabels(nb_, fontsize=8)
ax[1].set_ylabel("matched-AUROC under label permutation")
ax[1].set_title("X3 : null must be 0.5 by construction", fontsize=9)
ax[1].grid(alpha=.3, axis="y")

# ③ X5 — 두 팔이 같은 정도로 죽이나
lbl5 = ["k=0 (observed)", "detector error", "equal-variance noise"]
v5 = [X5["k0"], X5["검출기 오차(V2)"]["k1"], X5["동등분산 무작위"]["k1"]]
ax[2].bar(range(3), [v - NEW_NULL for v in v5],
          color=["tab:blue", "tab:red", "tab:orange"])
ax[2].axhline(0, color="k", lw=1.0)
ax[2].set_xticks(range(3)); ax[2].set_xticklabels(lbl5, fontsize=7, rotation=12)
ax[2].set_ylabel("S3 excess over measured null")
ax[2].set_title("X5 : is detector error special?", fontsize=9)
ax[2].grid(alpha=.3, axis="y")
fig.tight_layout()
PNG = run.save_fig("q7x_diagnostics", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in ("X0", "X1", "X1b", "X2", "X3", "X4", "X5"):
    run.log(f"  {g:<5}{VERD.get(g, '(미실행)')}")
run.log("")
run.log(f"  ★★★ **X1 판정 = {X1['shape']}**")
if X1["shape"] == "SNR 한계":
    run.log("     → 점 통계량이 잡음에 지배된다. **Q7-Y 기본형 = 적분·매치드필터.**")
    run.log("       개인 P 템플릿(그 레코드 N 비트) × 구간 정규화 상호상관 **최댓값** —")
    run.log("       잡음을 평균하고(SNR↑) 위치도 불변이며 **딜리니에이터가 필요 없다**")
elif X1["shape"] == "지터 한계":
    run.log("     → **Q7-Y 기본형 = 이동 불변 표현.** 구간 최대 정규화 상호상관 ·")
    run.log("       자기상관 · max-pooling — 위치를 특징에서 제거한다")
else:
    run.log("     → ⛔ **위치를 흔들어도 λ 가 안 떨어진다.** Q7-V 의 λ 0.2586 은 위치에서 온")
    run.log("       게 아니다. **Q7-Y 를 짓기 전에** 출처를 찾아야 한다 — X2(꼬리)·X0(고전")
    run.log("       오차 위배)·전처리 불일치 순으로 본다")
run.log("")
if no_("X3"):
    run.log("  ⛔ **쌍 내 치환인데도 null 이 0.5 가 아니다 — S3 추정량에 버그가 있다.**")
    run.log("     아래 어떤 숫자도 확정으로 쓰지 않고 S3 를 먼저 다시 본다")
else:
    run.log(f"  ★ 관문 문턱을 **{NEW_NULL:.4f}** 로 교정했다(0.5 가 아니다)")
if no_("X5"):
    run.log("  ★ **V2 의 기울기는 인용하지 않는다** — 동등분산 잡음도 같이 죽인다.")
    run.log("    「자가 병목」의 근거는 **V1(λ) + X1(λ 를 올릴 수 있나)** 뿐이다")
run.log("")
run.log("  ▸ 이 런은 **새 데이터를 쓰지 않았고 SVEB 질문에도 답하지 않는다** — 도구를 쟀다")
run.log("  ▸ 딥러닝 판단은 **X6 의 W1 상한**으로 한다. S3 의 탈감쇠는 **기전**의 크기다")

run.finish({
    "exp_id": "quest46_q7x_diagnostics",
    "metric": "lambda_at_key_delta",
    "value": float(X1[D_KEY]["p_score"]["lam"]),
    "passed": bool(ok_("X1") and ok_("X3")),
    "summary": ("λ 가 위치에서 오는지(지터) 통계량 자체에서 오는지(SNR) 가르고, S3 의 null 이 "
                "왜 0.5 가 아닌지 쌍 내 치환으로 확인하고, 검출기 오차가 특별한지 동등분산 "
                "잡음과 비교한다. 결과가 Q7-Y 설계를 정한다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "X0": CONFIG.get("X0", {}), "X1": CONFIG.get("X1", {}), "X2": CONFIG.get("X2", {}),
    "X3": CONFIG.get("X3", {}), "X4": CONFIG.get("X4", {}), "X5": CONFIG.get("X5", {}),
    "X6": CONFIG.get("X6", {}), "new_null": float(NEW_NULL),
    "but": CONFIG.get("but", {}), "svdb": CONFIG.get("svdb", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step lambda-diagnostics`")